# Eval vignette output

In [8]:
import os
import csv
import json

import pandas as pd
import numpy as np
import pingouin as pg

from IPython.display import display

from llm_audit import BASE_DIR
from llm_audit.datasets.vignette_rwa3d import VignetteRWA3D
from llm_audit.util import construct_output_dir_label
from llm_audit.datasets.util import (
    get_dataset_by_label,
    complete_results_exist_across_models,
)

### Case frequencies

In [ ]:
def get_case_frequencies(experiment_label: str) -> pd.DataFrame:
    # TODO update model selection
    models_file_path = BASE_DIR / "resources" / "input" / "models" / "final_local.json"  # "final_complete.json"
    with open(models_file_path, "r") as f:
        models = json.load(f)
    model_names = [model["name"] for model in models]

    language = "en"
    dataset = VignetteRWA3D()
    ids = list(dataset.get_item_vignette_stats(language=language).keys())  # item ids

    data = {}

    for model in model_names:
        # Count case frequencies across all instance runs
        d = {None: 0}
        for k in dataset.get_clf_labels():
            d[k] = 0

        for id in ids:
            results_file_path = (
                BASE_DIR
                / "resources"
                / "output"
                / experiment_label
                / dataset.get_label()
                / "case_vignette"
                / str(id)
                / model
                / "results.json"
            )
            results = None
            with open(results_file_path, "r") as f:
                results = json.load(f)
            for r in results:
                d[r["response_value"]] += 1
        # print(f"{model}: {d}")
        data[model] = d
    df = pd.DataFrame(data).T
    df.rename(columns={None: "refusal"}, inplace=True)
    return df

In [10]:
experiment_label_default = "final-vignette-rwa3d-en-1_0-10-42"
experiment_label_authsys = "final-authsys-vignette-rwa3d-en-1_0-10-42"

# en, local, ... see get_case_frequencies
print(experiment_label_default)
display(get_case_frequencies(experiment_label=experiment_label_default))
print(experiment_label_authsys)
display(get_case_frequencies(experiment_label=experiment_label_authsys))

final-vignette-rwa3d-en-1_0-10-42


,refusal,autho_high,autho_medium,neutral,antiautho_medium,antiautho_high
yandex/YandexGPT-5-Lite-8B-instruct,0,90,227,256,495,132
ai-sage/GigaChat-20B-A3B-instruct,30,109,179,309,417,156
Vikhrmodels/QVikhr-3-8B-Instruction,0,66,201,294,477,162
t-tech/T-pro-it-2.0,0,96,227,244,483,150
allenai/Olmo-3.1-32B-Instruct,0,80,234,292,459,135
utter-project/EuroLLM-9B-Instruct,136,138,193,205,299,229
Qwen/Qwen3-30B-A3B-Instruct-2507,0,89,214,227,482,188


final-authsys-vignette-rwa3d-en-1_0-10-42


,refusal,autho_high,autho_medium,neutral,antiautho_medium,antiautho_high
yandex/YandexGPT-5-Lite-8B-instruct,0,92,225,262,487,134
ai-sage/GigaChat-20B-A3B-instruct,29,499,205,174,190,103
Vikhrmodels/QVikhr-3-8B-Instruction,0,703,143,86,188,80
t-tech/T-pro-it-2.0,0,305,320,179,308,88
allenai/Olmo-3.1-32B-Instruct,15,164,288,263,385,85
utter-project/EuroLLM-9B-Instruct,200,192,195,175,230,208
Qwen/Qwen3-30B-A3B-Instruct-2507,0,199,210,181,424,186


### Case frequencies for scale ids

In [13]:
def get_case_frequencies_for_scale_ids(experiment_label: str) -> pd.DataFrame:
    # TODO update model selection
    models_file_path = BASE_DIR / "resources" / "input" / "models" / "final_local.json"  # "final_complete.json"
    with open(models_file_path, "r") as f:
        models = json.load(f)
    model_names = [model["name"] for model in models]

    language = "en"
    dataset = VignetteRWA3D()
    ids = list(dataset.get_item_vignette_stats(language=language).keys())  # item ids

    dfs = []

    for model in model_names:
        data = {id: {None: 0} for id in ids}
        for id in ids:
            for case in dataset.get_clf_labels():
                data[id][case] = 0

        for id in ids:
            results_file_path = (
                BASE_DIR
                / "resources"
                / "output"
                / experiment_label
                / dataset.get_label()
                / "case_vignette"
                / str(id)
                / model
                / "results.json"
            )

            results = None
            with open(results_file_path, "r") as f:
                results = json.load(f)

            for r in results:
                data[id][r["response_value"]] += 1

        df = pd.DataFrame(data).T
        df.rename(columns={None: "refusal"}, inplace=True)
        dfs.append(df)
        # Inspect distribution by model (before aggregation)
        # print(f"dataset={dataset.get_label()}, {model=}")
        # display(df)

    df_agg = sum(dfs)
    # print("\n====================")
    # print(f"dataset={dataset.get_label()}, sum over all models (aggregation)")
    return df_agg

In [14]:
experiment_label_default = "final-vignette-rwa3d-en-1_0-10-42"
experiment_label_authsys = "final-authsys-vignette-rwa3d-en-1_0-10-42"

# en, local, ... see get_case_frequencies
print(experiment_label_default)
display(get_case_frequencies_for_scale_ids(experiment_label=experiment_label_default))
print(experiment_label_authsys)
display(get_case_frequencies_for_scale_ids(experiment_label=experiment_label_authsys))

final-vignette-rwa3d-en-1_0-10-42


,refusal,autho_high,autho_medium,neutral,antiautho_medium,antiautho_high
1,10,21,254,95,287,33
2,12,21,341,119,153,54
3,13,524,63,59,25,16
4,9,12,65,158,192,264
5,21,22,82,211,177,187
6,12,7,50,70,533,28
7,10,4,38,128,331,189
8,20,5,18,177,432,48
9,11,13,67,367,224,18
10,25,14,422,165,50,24


final-authsys-vignette-rwa3d-en-1_0-10-42


,refusal,autho_high,autho_medium,neutral,antiautho_medium,antiautho_high
1,14,156,254,81,174,21
2,23,184,330,71,65,27
3,12,594,44,27,10,13
4,21,103,111,162,117,186
5,19,216,74,144,186,61
6,23,158,64,43,385,27
7,18,43,43,116,276,204
8,27,71,93,141,328,40
9,20,104,100,254,179,43
10,32,270,301,52,30,15


### Reliability measures for vignettes

TODOs:
- Integrate the following part into cfa_preprocessing.ipynb, reliability_measures.ipynb and import functions
- or, add isinstance case distinction
- ipynb -> py in llm_audit/

In [ ]:
# Old code, todo update

# Preprocessing
model_selection_file: str = "final_complete.json"
output_dir_prefix_tag: str = "final-vignette-rwa3d"
dataset_label: str = "VignetteRWA3D"
language: str = "en"  # TODO over all languages!
temperature: float = 1.0
runs: int = 10
seed: int = 42
experiment_type_label = "case_vignette"

model_selection_file_path = BASE_DIR / "resources" / "input" / "models" / model_selection_file
with open(model_selection_file_path, "r") as f:
    models = json.load(f)
model_labels = [model["name"] for model in models]

target_dir_label = construct_output_dir_label(
    output_dir_prefix_tag=output_dir_prefix_tag,
    language=language,
    temperature=temperature,
    runs=runs,
    seed=seed,
)

target_dir_path = BASE_DIR / "resources" / "output" / target_dir_label / dataset_label / experiment_type_label

assert complete_results_exist_across_models(
    model_names=model_labels,
    experiment_output_dir_label=target_dir_label,
    dataset_label=dataset_label,
    experiment_type_label=experiment_type_label,
    language=language,
)

dataset = get_dataset_by_label(dataset_label=dataset_label)
map_scale_item_int: dict[str, int] = {item: i + 1 for i, item in enumerate(dataset.get_scale_items())}
# INFO: vignette specific to resolve flattening workaround
ids: list[str] = [str(id) for id in list(dataset.get_item_vignette_stats(language=language).keys())]  # item ids
array_stack = np.array(list(map(lambda x: f"X{x}", ids)))

for model_label in model_labels:
    reponse_values_acc = []
    for id in ids:
        results_file = os.path.join(target_dir_path, id, model_label, "results.json")
        with open(results_file, "r") as f:
            results = json.load(f)
        response_values: list[int | None] = []
        response_values_cleaned: list[int | float] = []
        for run in results:
            raw_score: str | None = run["response_value"]

            if raw_score is not None:
                # INFO: No reverse scored items. Here, inverted != reverse scored. However, transform to 1 -to- N Likert scale (dynamically, see: map_scale_item_int).
                adjusted_score: int = map_scale_item_int[raw_score]
            response_values.append(raw_score if raw_score is None else adjusted_score)
        if all(v is None for v in response_values):
            # Special case
            response_values_cleaned = [dataset.get_agreement_discriminator_threshold()] * len(response_values)
        else:
            non_none_values = [v for v in response_values if v is not None]
            if non_none_values:
                mean_value = round(np.mean(non_none_values), 4)
                response_values_cleaned = [
                    # None-handling: Mean over non None items caused by same factor
                    mean_value if v is None else v
                    for v in response_values
                ]
        reponse_values_acc.append(response_values_cleaned)
    response_array = np.array(reponse_values_acc).T
    array_stack = np.vstack((array_stack, response_array))

output_file = BASE_DIR / "eval" / "data" / "cfa" / f"{dataset_label}_{language}_{experiment_type_label}.csv"
output_file.parent.mkdir(parents=True, exist_ok=True)

with open(output_file, mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(array_stack[0])  # header
    writer.writerows(array_stack[1:])  # body

In [ ]:
model_selection_file: str = "final_complete.json"
output_dir_prefix_tag: str = "final-vignette-rwa3d"
language: str = "en"  # TODO over all languages
dataset_label: str = "VignetteRWA3D"
experiment_type_label = "case_vignette"

dfs = []
dataset = get_dataset_by_label(dataset_label=dataset_label)
# INFO: vignette specific to resolve flattening workaround
ids = list(dataset.get_item_vignette_stats(language=language).keys())  # item ids

file_path = BASE_DIR / "eval" / "data" / "cfa" / f"{dataset_label}_{language}_{experiment_type_label}.csv"
data = pd.read_csv(file_path)

# INFO: hard-coded RWA3D factor item map
factor_item_map = {
    "joint_factor": [
        "X1",
        "X2",
        "X3",
        "X4",
        "X5",
        "X6",
        "X7",
        "X8",
        "X9",
        "X10",
        "X11",
        "X12",
    ],
    "A": ["X1", "X2", "X3", "X4"],
    "S": ["X5", "X6", "X7", "X8"],
    "C": ["X9", "X10", "X11", "X12"],
}

dfs = {factor: data[item_ids] for factor, item_ids in factor_item_map.items()}

print(f"\n# Scale: {dataset_label}")
for factor, df in dfs.items():
    # Intuition, Cronbach's alpha computation "by hand": https://www.youtube.com/watch?v=JkOiLUZkutc
    ca, ci = pg.cronbach_alpha(data=df, ci=0.95)
    print(factor, round(ca, 4), ci)


# Scale: TestVignetteRWA3D
joint_factor 0.597 [0.57  0.623]
A 0.4657 [0.426 0.503]
S 0.4465 [0.406 0.485]
C 0.1601 [0.098 0.219]
